# Amortized policy improvement, round 1: does a conservative value head over EI beat EI?

## Description

<!-- Written 2026-11-11 before the first run (pre-registration). Design: research_log/decisions/0003. Issue #13. -->

**Question.** Round 1 of approximate policy iteration from EI (0003 §3): a value head predicts, for every candidate query, its advantage over EI's own pick. The advantage is measured as the ground-truth final regret of querying the candidate and then following EI to the budget. Acting on that head with the conservative rule improves the policy by one step. Does the result beat EI on fresh tasks, and by how much? The answer says whether one improvement step captures a worthwhile planning gap on the PFN's own prior, at EI's cost per step.

**Setup.**
- *Base policy* π₀: EI on the frozen `pfn_large`, on the bo pipeline's candidate generator (1024 shifted Sobol points + 3 × 64 incumbent perturbations). The prior is noise-free (P0), so latent EI = EI.
- *Budget*: n_init = 2d shared Sobol points, T = 10d steps; d = 2 in this pilot.
- *Labels* (0003 §5.2), on 4000 split-4 instances:
  - one behaviour rollout per instance: EI for 80%, UCB(0.99) or PI for 10% each, for off-policy coverage;
  - one branch step t ~ U{0..T−1} per rollout;
  - a candidate set of 133: EI's pick x₀; the argmaxes of UCB 0.9 / 0.99, PI and the mean; 64 shifted Sobol points; 4 × 16 perturbations of the 4 best points at σ ∈ {0.01, 0.03, 0.1};
  - 16 labeled candidates: x₀, the heuristic argmaxes, the top 4 by EI, then random;
  - each labeled candidate is queried and then continued with EI to T, under 4 continuation replicates shared across candidates (common random numbers);
  - label = mean over replicates of R(x₀) − R(c), with R the final regret in % of the task's gap f_med − f\*, plus its standard error.
- *Value head* (0003 §4.4):
  - inputs: the PFN's final query-token state z(x), bar-distribution summaries relative to the incumbent, distances, source, within-set ranks, and global budget / belief features;
  - architecture: an MLP, then FiLM from the global features, then 2 set-transformer layers;
  - outputs: Â anchored at x₀ (Â(x₀) = 0) and σ̂;
  - training: Gaussian NLL with the label's standard error added to the predicted variance; 10% of instances held out for early stopping.
- *Conservative rule* (0003 §3.3): query argmax (Â − c σ̂) only if that maximum is > 0, otherwise x₀.
- *Calibration*: c ∈ {0, 0.5, 1, 2, 3, 4} on 300 split-0 tasks, each paired against EI. c\* = the c with the lowest mean normalized regret (ties go to the larger c). It is fixed before the test run.
- *Test*: 1000 split-1 tasks. Compared: EI; π₁ at c\* (the pre-registered comparison); π₁ at the other c (secondary; not selected on).
- *Null control*: a head trained identically on labels permuted across the candidates of each state, run at c\*. It carries no candidate information, so it must not beat EI. How much it loses shows how well the rule protects.
- Everything is paired on tasks, with common random numbers; fp32.

*Revision after the sanity run (2026-11-11, before the pilot):* the grid of c was {0, 0.5, 1, 2} as in 0003 §5.4. With a 180-state head in the sanity run, every c in it was worse than EI on calibration, and c\* is chosen only within the grid. The grid was therefore extended by 3 and 4, so that a weak head can be made more conservative. Nothing else changed.

**What to look for.**
1. The primary effect: the paired difference Δ = mean_task [R(π₁, c\*) − R(EI)], with its bootstrap 95% CI, absolute and relative to EI's mean R. Secondary: the gain on log₁₀ regret (the project's usual measure).
2. The null head's Δ.
3. The deviation rate per c: how often π₁ leaves EI's pick, and from which candidate sources.
4. Validation fit of the head: Â against the label, per branch step.
5. The cross-fitted *clairvoyant* advantage map: the best labeled candidate is selected on 2 continuation replicates and scored on the other 2. It is the reachable first-order gain within this candidate class, as a diagnostic only. It uses the true task, so it is an upper reference, never a result.

**GO / NO-GO** (rule `analysis/api.py::api_verdict`, worthwhile = 5% relative):
- **INVALID** if the null head beats EI with its CI below 0.
- **GO** if π₁ at c\* lowers mean normalized regret by ≥ 5% relative with the CI of Δ below 0.
- **NO-GO** if the CI rules out a 5% relative gain.
- **INCONCLUSIVE** otherwise.

GO means a confirm run at d ∈ {2, 4} with the same rule, and then round 2. NO-GO at d = 2 means one improvement step from EI on P0 is not worth it at this budget. That supports 0003 §3.6 (EI near a fixed point of policy iteration for this prior), but only for this candidate class and readout.

**Limits.**
- `pfn_large` is valid for n ≤ 100 and d ≤ 6; here the context is ≤ 12d = 24.
- The prior is P0: noise-free and on the global ECDF scale. The noise awareness and budget awareness motivated in 0003 are therefore barely testable; only myopia and PFN-error correction remain.
- One fixed T per d: the budget input is t/T only.
- Deviations from 0003, all for cost or because the components don't exist yet: no two-headed PFN (Stage 0); no context-token pooling in the head; features computed once and kept in memory, not recomputed on load; EI's pick comes from the fixed Sobol + perturbation generator, not Adam refinement; no carry-over candidates; a 4000-state pilot instead of 50k.
- The calibration set (300 tasks) is small, so c\* is noisy. All c are reported on the test split for that reason, but only c\* is the claim.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/api_round1/<sweep_id>"  # set after `just snapshot runs/api_round1/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.